In [2]:
import os
import re

import torch
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import nltk
import sys 
import analysis_utils
sys.path.append("../")

from checkpoint import CheckPoint
from datasets import LetterStringDataLoader
import generate_data

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def bootstrapped_confint(arr, n_samples:int = 10_000, alpha:float=0.05):
    # sample with replacement n_samples times and take the mean of each sample:
    samples = np.array([np.mean(np.random.choice(arr, size=len(arr), replace=True)) for _ in range(n_samples)])
    # sort means in ascending order:
    samples.sort()
    # get bootstrapped confidence interval boundaries:
    confint_low = samples[int(n_samples * (alpha / 2))]
    confint_upp = samples[int(n_samples * (1 - alpha / 2))]
    return confint_low, confint_upp

In [29]:
# batching experiments no copy tasks, 20 permuted training alphabets:
dir_path = "../models/batching_experiments"
folder = os.listdir(dir_path)
cps_perm20 = []

for filename in folder:
    cp = CheckPoint.from_pt("/".join([dir_path, filename]))
    cp.train_config.filename_model = filename
    cp.num_perm_alphs = 20
    cps_perm20.append(cp)


# batching experiments with copy tasks, 20 permuted training alphabets:
dir_path = "../models/copy_batching_experiments"
folder = os.listdir(dir_path)
cps_copy_perm20 = []

for filename in folder:
    cp = CheckPoint.from_pt("/".join([dir_path, filename]))
    cp.train_config.filename_model = filename
    cp.num_perm_alphs = 20
    cps_copy_perm20.append(cp)


# batching experiments with copy tasks, 200 permuted training alphabets:
dir_path = "../models/num_permuted_alphabets"
folder = os.listdir(dir_path)
cps_copy_perm200 = []

for filename in folder:
    if "200" in filename:
        cp = CheckPoint.from_pt("/".join([dir_path,filename]))
        cp.train_config.filename_model = filename
        cp.num_perm_alphs = 200
        cps_copy_perm200.append(cp)

In [30]:
for cp in cps_copy_perm20:
    print(cp.train_config.filename_model)

MLC_batchalph_dallstudy1_copy_perm20_nep20.pt
MLC_batchalph_dallstudy1_copy_perm20_nep20_rep1.pt
MLC_batchalph_dallstudy1_copy_perm20_nep20_rep2.pt
MLC_batchalph_dallstudy1_copy_perm20_nep20_rep3.pt
MLC_batchalph_dallstudy1_copy_perm20_nep20_rep4.pt
MLC_batchbyboth_dallstudy1_copy_perm20_nep20.pt
MLC_batchbyboth_dallstudy1_copy_perm20_nep20_rep1.pt
MLC_batchbyboth_dallstudy1_copy_perm20_nep20_rep2.pt
MLC_batchbyboth_dallstudy1_copy_perm20_nep20_rep3.pt
MLC_batchbyboth_dallstudy1_copy_perm20_nep20_rep4.pt
MLC_batchrand_dallstudy1_copy_perm20_nep20.pt
MLC_batchrand_dallstudy1_copy_perm20_nep20_rep1.pt
MLC_batchrand_dallstudy1_copy_perm20_nep20_rep2.pt
MLC_batchrand_dallstudy1_copy_perm20_nep20_rep3.pt
MLC_batchrand_dallstudy1_copy_perm20_nep20_rep4.pt
MLC_batchtrans_dallstudy1_copy_perm20_nep20.pt
MLC_batchtrans_dallstudy1_copy_perm20_nep20_rep1.pt
MLC_batchtrans_dallstudy1_copy_perm20_nep20_rep2.pt
MLC_batchtrans_dallstudy1_copy_perm20_nep20_rep3.pt
MLC_batchtrans_dallstudy1_copy_perm20

In [24]:
def get_accuracy_table(checkpoints:list):
    # set up dataset to store accuracies:
    all_accs = pd.DataFrame(
        columns=["filename_model", "batching_method", "num. seen alphabets in training", "seen transform.", "new transform.", "alphabets"]
    )
    # new alphabets are constant across all checkpoints
    test_new_alph = LetterStringDataLoader(
        mode="test", 
        data_dir="../data/all_transformations_study1_new_alphabets",
        batch_size=2000
    )
    for cp in checkpoints:
        model = cp.load_model(verbose=False)

        # obtain accuracies on seen training alphabets:
        test_seen_alph = LetterStringDataLoader(
            mode="test", 
            data_dir=f"../{cp.train_config.dir_data}",
            batch_size=2000
        )
        pred = analysis_utils.predict_dataset(test_seen_alph, model, alternative_rule_errors=False)
        # exclude standard alphabet:
        pred = pred[pred["n_perm"]!= 0]
        test_acc_seen_transform = np.mean(pred[pred.distribution == "in"]["correct"])
        test_acc_new_transform = np.mean(pred[pred.distribution == "out-of"]["correct"])
        all_accs.loc[all_accs.shape[0],:] = [
            cp.train_config.filename_model, 
            cp.train_config.batching_method, 
            cp.num_perm_alphs, 
            test_acc_seen_transform, 
            test_acc_new_transform,
            "seen"
        ]

        # obtain accuracies on new alphabets:
        pred = analysis_utils.predict_dataset(test_new_alph, model, alternative_rule_errors=False)
        # exclude standard alphabet:
        pred = pred[pred["n_perm"]!= 0]
        test_acc_seen_transform = np.mean(pred[pred.distribution == "in"]["correct"])
        test_acc_new_transform = np.mean(pred[pred.distribution == "out-of"]["correct"])
        all_accs.loc[all_accs.shape[0],:] = [
            cp.train_config.filename_model, 
            cp.train_config.batching_method, 
            cp.num_perm_alphs, 
            test_acc_seen_transform, 
            test_acc_new_transform,
            "new"
        ]
    return all_accs

In [ ]:
tbl_copy_perm200 = get_accuracy_table(cps_copy_perm200)
tbl_copy_perm200.to_csv("copy_perm200_accuracies.csv", index=False)
tbl_copy_perm200

,filename_model,batching_method,num. seen alphabets in training,seen transform.,new transform.,alphabets
0,MLC_batchalph_dallstudy1_copy_perm200_nep20.pt,alphabet,200,0.99992,0.361119,seen
1,MLC_batchalph_dallstudy1_copy_perm200_nep20.pt,alphabet,200,0.998728,0.262257,new
2,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.893468,0.31361,seen
3,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.679281,0.036531,new
4,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,1.0,0.327959,seen
5,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.999682,0.194963,new
6,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,1.0,0.286832,seen
7,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.993958,0.098058,new
8,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.999761,0.326567,seen
9,MLC_batchalph_dallstudy1_copy_perm200_nep20_re...,alphabet,200,0.99841,0.147664,new


In [ ]:
cps_copy_perm20

In [31]:
tbl_copy_perm20 = get_accuracy_table(cps_copy_perm20)
tbl_copy_perm20.to_csv("copy_perm20_accuracies.csv", index=False)
tbl_copy_perm20

Generating predictions:  32%|███▏      | 7/22 [05:19<11:25, 45.71s/it]


KeyboardInterrupt: 

In [ ]:
tbl_perm20 = get_accuracy_table(cps_copy_perm20)
tbl_perm20.to_csv("perm20_accuracies.csv", index=False)
tbl_perm20